<a href="https://colab.research.google.com/github/megagnom111/kursovaya3/blob/main/KursovayaAAA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Курсовая работа Алексеевой А. А. 318

# Анализ периодов булевых функций по алгоритму A1

## Курсовая работа

**Студент:** Алексеева Алёна Алексеевна

**Группа:** 318

**Научный руководитель:** Селезнёва С.Н.

**Дата:** 2026

---

## Постановка задачи

Нам дан **S-блок (перестановка)** шифра. Из этого S-блока мы получаем
8 булевых функций (по числу выходных бит).

**Цель работы:** определить, является ли шифр **уязвимым** для криптоанализа
на основе линейной структуры.

**Метод:** для каждой из 8 функций находим **базис пространства периодов**
с помощью алгоритма A1. Затем ищем **общие периоды** для всех 8 функций.

---

## Что такое период булевой функции?

**Определение:** Вектор $a = (a_1, ..., a_n) \in \{0,1\}^n$ называется **периодом**
функции $f(x_1, ..., x_n)$, если для всех $x \in \{0,1\}^n$ выполняется:

$$
f(x_1 \oplus a_1, ..., x_n \oplus a_n) = f(x_1, ..., x_n)
$$

где $\oplus$ — сложение по модулю 2 (XOR).

**Пример:** Для функции $f(x_1, x_2) = x_1$ вектор $(0,1)$ является периодом,
так как $f(x_1, x_2 \oplus 1) = x_1 = f(x_1, x_2)$.

---

## Криптографическое значение

Наличие ненулевых периодов у булевых функций, используемых в шифрах,
создаёт **линейную структуру**, которая может быть использована для
криптоанализа. Чем больше периодов — тем уязвимее шифр.

---
## Связь с уязвимостью шифра

**Ключевая идея:** Если у всех 8 функций шифра **есть общий ненулевой период**,
то существует сдвиг ключа/сообщения, который **не меняет выход шифра**.
Это делает шифр уязвимым для атак.

---

## Структура работы

1. Получение 8 булевых функций из S-блока
2. Построение полиномов Жегалкина для каждой функции
3. Применение алгоритма A1 для поиска периодов
4. Нахождение общих периодов для всех 8 функций
5. Вывод об уязвимости шифра

---

# Часть 1. Установка и импорт библиотек, конфигурация констант.

In [33]:
!pip install galois numpy matplotlib -q

import galois
import numpy as np
from math import log
from typing import Optional
# import matplotlib.pyplot as plt

print("✅ Библиотеки загружены!")

✅ Библиотеки загружены!


In [3]:
# количество функций = количество бит в числах
FUNC_COUNT = None
# длина вектора значений функций = количество чисел
POLINOM_LEN = None

# Часть 2. Построение полиномов Жегалкина для зашифрованных функций.

## По шифру строим таблицу истинности, где получаем определённое количество функций. Для каждой функции строим полином Жегалкина и вектор коэффицентов.

## 1. Построение таблицы истинности

In [4]:
def create_truth_table(permutation: list[int]) -> list[list[int]]:
    """
    Args:
        permutation: список целых чисел длиной POLINOM_LEN,
                    значения от 0 до POLINOM_LEN
                    ( список шифрования )

    Return:
        список из FUNC_COUNT списков, каждый содержит POLINOM_LEN бит
        ( список векторов значений функций )
        -> list[list[int]]

    """

    columns = [[] for _ in range(FUNC_COUNT)]

    for x in range(POLINOM_LEN):
        y = permutation[x]
        binary = format(y, f'0{FUNC_COUNT}b')

        for bit_pos in range(FUNC_COUNT):
            columns[bit_pos].append(int(binary[bit_pos]))

    return columns

Пример работы данной функции на примере шифра [4, 3, 5, 1, 2, 6, 7, 0]

In [11]:
print("=" * 60)
print("ПРИМЕР: Получения таблицы истинности и векторов функций из S-блока")
print("=" * 60)

example_sbox = [4, 3, 5, 1, 2, 6, 7, 0]
FUNC_COUNT = 3
POLINOM_LEN = 8
print(f"Шифр: {example_sbox}")
print(f"Количество функций: {FUNC_COUNT}")
print(f"Количество входов: {POLINOM_LEN}")
print()

columns = create_truth_table(example_sbox)

print("Таблица истинности:")
for i in range(POLINOM_LEN):
    print(f'{i:3}  | ', end='')
    for col in range(FUNC_COUNT):
        print(f'{columns[col][i]} ', end='')
    print("\n")

print()
print("Векторы значений функций:")
for i, col in enumerate(columns):
    print(f"  F{i+1}: {col}")


ПРИМЕР: Получения таблицы истинности и векторов функций из S-блока
S-блок: [4, 3, 5, 1, 2, 6, 7, 0]
Количество функций: 3
Количество входов: 8

Таблица истинности:
  0  | 1 0 0 

  1  | 0 1 1 

  2  | 1 0 1 

  3  | 0 0 1 

  4  | 0 1 0 

  5  | 1 1 0 

  6  | 1 1 1 

  7  | 0 0 0 


Векторы значений функций:
  F1: [1, 0, 1, 0, 0, 1, 1, 0]
  F2: [0, 1, 0, 0, 1, 1, 1, 0]
  F3: [0, 1, 1, 1, 0, 0, 1, 0]


## 2. Построение полиномов Жегалкина для каждой функции

In [12]:
def fast_transform(vector: list[int]) -> list[int]:
    """
    Реализация быстрого алгоритма построения вектора полинома Жегалкина
    Args:
        vector: бинарный вектор значений функции длины POLINOM_LEN
    Result:
        бинарный вектор полинома Жегалкина длины POLINOM_LEN
    """
    n = len(vector)
    a = vector.copy()

    step = 1
    while step < n:
        for i in range(0, n, step * 2):
            for j in range(i, i + step, 1):
                a[j + step] ^= a[j]
        step *= 2

    return a

In [13]:
def format_zhegalkin_term(coeff_index: int) -> str:
    """
    Функция для форматирования терм полинома Жегалкина
    Args:
        coeff_index: целое число от 0 до POLINOM_LEN
    Result:
        строка - представление терма, например, "x1*x3" или "1"
    """
    if coeff_index == 0:
        return "1"

    bin_coeff = format(coeff_index, f'0{FUNC_COUNT}b')
    term = []

    for i, bit in enumerate(bin_coeff):
        if bit == "1":
            term.append(f"x{i + 1}")

    return "*".join(term) if term else "1"

In [20]:
def zhegalkin_polinom(coeffs: list[int], func_index = 0) -> str:
    non_zero_terms = []
    for i, coeff in enumerate(coeffs):
        if coeff == 1:
            term = format_zhegalkin_term(i)
            non_zero_terms.append(term)

    if not non_zero_terms:
        return f"F{func_index + 1} = 0"
    else:
        return f"F{func_index + 1} = {" ⊕ ".join(non_zero_terms)}"

In [21]:
print("=" * 60)
print("ПРИМЕР: Построение полинома Жегалкина")
print("=" * 60)

# Функция F1 из предыдущего примера
values_f1 = [1, 0, 1, 0, 0, 1, 1, 0]
print(f"Вектор значений F1: {values_f1}")

coeffs_f1 = fast_transform(values_f1)
print(f"Вектор коэффициентов: {coeffs_f1}")

polynomial = zhegalkin_polinom(coeffs_f1)
print(f"Полином Жегалкина: {polynomial}")

print("\nПроверка (ручной расчёт):")
print("  Полином должен быть: 1 ⊕ x3 ⊕ x1 ⊕ x1x2")

ПРИМЕР: Построение полинома Жегалкина
Вектор значений F1: [1, 0, 1, 0, 0, 1, 1, 0]
Вектор коэффициентов: [1, 1, 0, 0, 1, 0, 1, 0]
Полином Жегалкина: F1 = 1 ⊕ x3 ⊕ x1 ⊕ x1*x2

Проверка (ручной расчёт):
  Полином должен быть: 1 ⊕ x3 ⊕ x1 ⊕ x1x2


# Часть 3. Реализация алгоритма *А1*.

In [22]:
def get_max_weight_mask(coeffs: list[int]) -> int:
    """
    Шаг 2.1 алгоритма A1.

    Что делает:
        Выбирает среди всех ненулевых коэффициентов полинома тот,
        который соответствует моному:
        1) с наибольшим весом (количеством переменных в мономе)
        2) если веса равны — с наибольшим лексикографическим номером

    Почему это важно:
        Алгоритм начинает с самых "старших" мономов,
        чтобы постепенно понижать степень полинома.

    Аргументы:
        coeffs: вектор коэффициентов полинома (длина 2^n)

    Возвращает:
        маску выбранного монома (целое число)
    """
    max_weight = -1  # максимальный вес среди просмотренных
    best_mask = 0    # маска монома с максимальным весом (и номером)

    # Перебираем все возможные мономы (от 0 до 2^n-1)
    for mask, c in enumerate(coeffs):
        if c == 0:
            continue  # коэффициент = 0 — монома нет
        w = bin(mask).count('1')  # вес = количество переменных в мономе
        # Сравниваем: сначала по весу, потом по лексикографическому номеру
        if w > max_weight or (w == max_weight and mask > best_mask):
            max_weight = w
            best_mask = mask
    return best_mask


In [23]:
def c_value(coeffs: list[int], s_mask: int, i: int, j: int) -> int:
    """
    Вычисление коэффициента при замене переменной.

    Что делает:
        Вычисляет c_f(s - e_i + e_j) — коэффициент при мономе,
        полученном из монома s заменой переменной i на j.

    Зачем это нужно:
        При построении g_k мы смотрим, какие переменные j из o(s)
        "связаны" с переменной i через замену. Если коэффициент
        при таком заменённом мономе равен 1, то j входит в линейную форму.

    Аргументы:
        coeffs: вектор коэффициентов исходного полинома
        s_mask: маска монома s (исходный моном)
        i: номер заменяемой переменной (1-индексация)
        j: номер переменной, на которую заменяем (1-индексация)

    Возвращает:
        0 или 1 — коэффициент при заменённом мономе
    """
    # Проверяем условие: i должен быть в мономе s, j — не должен
    if ((s_mask >> (i - 1)) & 1) == 0 or ((s_mask >> (j - 1)) & 1) == 1:
        return 0

    # Строим маску нового монома: удаляем i, добавляем j
    new_mask = s_mask & ~(1 << (i - 1)) | (1 << (j - 1))
    return coeffs[new_mask] if new_mask < len(coeffs) else 0


In [24]:
def multiply_polinoms(p1: dict, p2: dict) -> dict:
    """
    Умножение полиномов в алгебре Жегалкина.

    Что делает:
        Перемножает два полинома, учитывая правило x² = x.

    Зачем это нужно:
        При построении g_k = ∏ (x_i + Σ c·x_j) нужно перемножить
        несколько линейных форм. Умножение выполняется в кольце
        полиномов Жегалкина, где x² = x (идемпотентность).

    Как работает:
        Для каждой пары мономов из p1 и p2:
        - Новый моном = XOR масок (симметрическая разность)
        - Коэффициент = AND коэффициентов (умножение в GF(2))
        - Складываем по модулю 2 (XOR) одинаковые мономы

    Аргументы:
        p1, p2: словари {маска: коэффициент} для двух полиномов

    Возвращает:
        словарь {маска: коэффициент} для произведения
    """
    res = {}
    for m1, c1 in p1.items():
        if not c1:
            continue
        for m2, c2 in p2.items():
            if not c2:
                continue
            m = m1 ^ m2  # XOR — объединение переменных с удалением дублей
            res[m] = res.get(m, 0) ^ (c1 & c2)  # сложение по модулю 2
            if res[m] == 0:
                del res[m]
    return res

In [25]:
def build_gk(coeffs: list[int], s_mask: int) -> dict:
    """
    Шаг 2.2 алгоритма A1.

    Что делает:
        Строит полином g_k = ∏_{i∈i(s)} (x_i + Σ_{j∈o(s)} c·x_j)

    Зачем это нужно:
        g_k — это полином, который содержит моном x^{s_k} (старший моном)
        и другие мономы меньшего веса. При вычитании g_k из f_k
        старший моном уничтожается, и степень полинома понижается.

    Как работает:
        1. Находим i(s) — переменные, входящие в моном s
        2. Находим o(s) — переменные, НЕ входящие в моном s
        3. Для каждого i ∈ i(s) строим линейную форму:
               x_i + Σ_{j∈o(s)} c_f(s - e_i + e_j)·x_j
        4. Перемножаем все эти линейные формы

    Аргументы:
        coeffs: вектор коэффициентов текущего полинома f_k
        s_mask: маска выбранного монома s_k

    Возвращает:
        словарь {маска: коэффициент} для полинома g_k
    """
    n = FUNC_COUNT

    # i(s) — переменные, входящие в моном (их биты = 1)
    i_list = [i+1 for i in range(n) if (s_mask >> i) & 1]
    # o(s) — переменные, НЕ входящие в моном (их биты = 0)
    o_list = [j+1 for j in range(n) if not ((s_mask >> j) & 1)]

    # Начинаем с полинома-единицы (1)
    result = {0: 1}

    # Для каждой переменной i из i(s) перемножаем скобки
    for i in i_list:
        # Строим линейную форму для текущего i
        linear = {1 << (i - 1): 1}  # начинаем с x_i

        # Добавляем слагаемые c·x_j для всех j из o(s)
        for j in o_list:
            if c_value(coeffs, s_mask, i, j):
                m = 1 << (j - 1)
                linear[m] = linear.get(m, 0) ^ 1  # XOR для сложения
                if linear[m] == 0:
                    del linear[m]

        # Умножаем накопленный результат на новую линейную форму
        result = multiply_polinoms(result, linear)

    return result

In [29]:
def algorithm_A1(coeffs: list[int]) -> list[list[int]]:
    """
    Алгоритм A1 (основной).

    Что делает:
        Строит систему линейных уравнений, решениями которой являются
        все периоды булевой функции f, заданной полиномом Жегалкина.

    Как работает (цикл, пока степень f_k >= 1):
        Шаг 2.1: Выбрать s_k — моном максимального веса
        Шаг 2.2: Построить g_k по формуле из статьи
        Шаг 2.3: h_k = f_k - g_k (вычитание в GF(2))
        Шаг 2.4: Добавить уравнения T_k из g_k в общую систему
        Шаг 2.5: f_{k+1} = h_k, повторить

    Почему это работает:
        По теореме 1 из статьи, период функции f является периодом
        каждого из полиномов g_k и h_k. Уравнения из g_k задают
        линейные условия на периоды. Пересечение всех этих условий
        даёт пространство всех периодов f.

    Аргументы:
        coeffs: вектор коэффициентов полинома Жегалкина (длина 2^n)

    Возвращает:
        список уравнений, каждое уравнение — список [c1, c2, ..., cn],
        означающий: c1·x1 + c2·x2 + ... + cn·xn = 0
    """
    n = FUNC_COUNT
    f_k = coeffs.copy()      # текущий полином (начинаем с исходного)
    equations = []           # накопленная система уравнений T

    # Основной цикл: пока есть мономы степени >= 1
    while True:
        # Проверяем, есть ли в f_k мономы степени >= 1
        degree_ge_1 = False
        for mask, c in enumerate(f_k):
            if c and bin(mask).count('1') >= 1:
                degree_ge_1 = True
                break
        if not degree_ge_1:
            break  # осталась только константа — выход из цикла

        # Шаг 2.1: выбираем s_k
        s_k = get_max_weight_mask(f_k)

        # Находим i(s_k) и o(s_k)
        i_list = [i+1 for i in range(n) if (s_k >> i) & 1]        # входящие
        o_list = [j+1 for j in range(n) if not ((s_k >> j) & 1)]  # не входящие

        # Шаг 2.2: строим g_k
        gk_dict = build_gk(f_k, s_k)

        # Шаг 2.3: h_k = f_k - g_k (в GF(2) вычитание = сложение)
        fk_dict = {m: 1 for m, c in enumerate(f_k) if c}  # f_k как словарь
        hk_dict = fk_dict.copy()
        for m, c in gk_dict.items():
            hk_dict[m] = hk_dict.get(m, 0) ^ c  # XOR
            if hk_dict[m] == 0:
                del hk_dict[m]

        # Обновляем f_k для следующей итерации
        f_k = [0] * len(coeffs)
        for m in hk_dict:
            f_k[m] = 1

        # Шаг 2.4: добавляем уравнения T_k в систему T
        # Каждое уравнение: x_i + Σ_{j∈o(s)} c·x_j = 0
        for i in i_list:
            eq = [0] * n
            eq[i - 1] = 1  # коэффициент при x_i
            for j in o_list:
                if c_value(coeffs, s_k, i, j):
                    eq[j - 1] ^= 1  # добавляем x_j
            equations.append(eq)

    return equations


In [32]:
print("=" * 60)
print("ПРИМЕР: Алгоритм A1 на функции с известным периодом")
print("=" * 60)

# Функция f = x₁x₂ ⊕ x₁x₃ (ожидаемый период (1,1,0))
coeffs = [0] * POLINOM_LEN
coeffs[0b110] = 1  # x₁x₂ (маска 6)
coeffs[0b101] = 1  # x₁x₃ (маска 5)

print(f"Полином: {zhegalkin_polinom(coeffs)}")
print("Ожидаемый период: (0,1,1)")
print()

equations = algorithm_A1(coeffs)

print(f"\nПолучено уравнений: {len(equations)}")
print("Система уравнений:")
for i, eq in enumerate(equations):
    terms = [f"x_{j+1}" for j, v in enumerate(eq) if v == 1]
    print(f"  {i+1}. {' + '.join(terms)} = 0")

ПРИМЕР: Алгоритм A1 на функции с известным периодом
Полином: F1 = x1*x3 ⊕ x1*x2
Ожидаемый период: (0,1,1)


Получено уравнений: 2
Система уравнений:
  1. x_1 + x_2 = 0
  2. x_3 = 0


#Часть 4. Нахождение базиса периодов для функции и нахождение общих периодов для нескольких функций

## Нахождение и вывод периодов для одной функции

In [34]:
def find_periods_basis(coeffs: list[int]) -> Optional[np.ndarray]:
    """
    Находит базис пространства периодов с помощью galois.

    Возвращает:
        - None: все векторы являются периодами (нет уравнений)
        - np.ndarray: матрица, строки которой — базисные векторы
    """
    equations = algorithm_A1(coeffs)

    if not equations:
        return None  # все векторы — периоды

    # Создаём поле GF(2)
    GF2 = galois.GF(2)

    # Преобразуем уравнения в матрицу (строки = уравнения, столбцы = переменные)
    A = GF2(equations)

    # Находим null space (пространство решений A·x = 0)
    null_space = A.null_space()

    return null_space

In [36]:
def reverse_vector(vec: list[int]) -> list[int]:
    """Переворачивает вектор (x₁ ↔ xₙ)."""
    return list(reversed(vec))


def print_periods_summary(basis: Optional[np.ndarray], func_name: str):
    """Выводит информацию о периодах функции."""
    if basis is None:
        print(f"{func_name}: ВСЕ ВЕКТОРЫ являются периодами (размерность = {FUNC_COUNT})")
    elif basis.shape[0] == 0 or (basis.shape[0] == 1 and np.all(basis[0] == 0)):
        print(f"{func_name}: только нулевой период")
    else:
        # Убираем нулевой вектор, если он есть
        nonzero_rows = [row for row in basis if not np.all(row == 0)]
        if not nonzero_rows:
            print(f"{func_name}: только нулевой период")
        else:
            print(f"{func_name}: размерность = {len(nonzero_rows)}")
            print(f"  Базис периодов:")
            for row in nonzero_rows:
                # Переворачиваем вектор для правильного отображения
                reversed_row = reverse_vector(row.tolist())
                print(f"    {reversed_row}")


In [42]:
# ПРИМЕР РАБОТЫ
print("=" * 60)
print("ПРИМЕР: Нахождение базиса периодов")
print("=" * 60)

coeffs = [0] * POLINOM_LEN
coeffs[0b110] = 1  # x₁x₂
coeffs[0b101] = 1  # x₁x₃

print(f"Полином: {zhegalkin_polinom(coeffs)}")
print()

basis = find_periods_basis(coeffs)
print_periods_summary(basis, 1)

print("\nПроверка: (0,1,1) действительно период?")
print("  f(0,1,1) = 0⊕0 = 0, f(0,0,0)=0 → да")

ПРИМЕР: Нахождение базиса периодов
Полином: F1 = x1*x3 ⊕ x1*x2

1: размерность = 1
  Базис периодов:
    [0, 1, 1]

Проверка: (0,1,1) действительно период?
  f(0,1,1) = 0⊕0 = 0, f(0,0,0)=0 → да


# Построение общего базиса для нескольких функций



In [43]:
def find_common_periods(all_coeffs: list[list[int]]) -> np.ndarray:
    """
    Находит периоды, общие для всех функций.

    Аргументы:
        all_coeffs: список векторов коэффициентов для каждой функции

    Возвращает:
        матрица, строки которой — базис общих периодов
    """
    all_equations = []

    for coeffs in all_coeffs:
        equations = algorithm_A1(coeffs)
        all_equations.extend(equations)

    if not all_equations:
        # Нет уравнений → все векторы являются общими периодами
        return None

    GF2 = galois.GF(2)
    A = GF2(all_equations)
    null_space = A.null_space()

    return null_space